# AxiomCart — Multi-Agent Voice Shopping Assistant
## Complete 9-Stage Build Walkthrough

**Stack:** LangGraph · LangChain · GPT-4o · ChromaDB · OpenAI Whisper/TTS  
**Estimated time:** ~4–4.5 hours end-to-end

```
FINAL ARCHITECTURE
==================

               ┌──────────────────────┐
  user query   │                      │
  ──────────►  │     orchestrator     │  ← classifies intent via Pydantic structured output
               │   (GPT-4o + Send())  │
               └───────────┬──────────┘
                            │
               ┌────────────┴─────────────┐
               │   parallel Send() fan-out │
               ▼                           ▼
  ┌─────────────────────┐    ┌──────────────────────┐
  │    product_agent    │    │    support_agent     │
  │   model ⇄ tools     │    │   model ⇄ tools      │
  │   (RAG catalog)     │    │   (order lookup,     │
  │                     │    │    escalation, HITL) │
  └────────┬────────────┘    └──────────┬───────────┘
           │                            │
           └─────────────┬──────────────┘
                         ▼
              ┌───────────────────────┐
              │      synthesizer      │  ← LLM merges multi-agent answers
              └───────────┬───────────┘
                          ▼
                     final_answer  ──► user (text or voice TTS)
```

> **Principle:** Each stage introduces exactly **one** new concept.  
> Every stage is a valid stopping point — Stage 4 alone covers RAG, tools, and agent subgraphs.

---
## Prerequisites — `uv` Virtual Environment

[`uv`](https://docs.astral.sh/uv/) is a Rust-based Python package manager.  
It replaces `pip` + `venv` + `pip-tools` with one tool that is 10–100× faster.

| | `venv` + `pip` | `uv` |
|---|---|---|
| Speed | network-bound | parallel downloads + global cache |
| Lockfile | manual | `uv lock` → `uv.lock` (hermetic) |
| Python pinning | manual | `uv python install 3.11` |
| PEP 517/518 | partial | full native support |

### Step 1 — Install `uv` (once per machine)

```bash
# macOS / Linux
curl -LsSf https://astral.sh/uv/install.sh | sh

# Windows (PowerShell)
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"

# Any platform via pip
pip install uv
```

In [1]:
import shutil, subprocess

# Check if uv is installed
if shutil.which("uv"):
    ver = subprocess.run(["uv", "--version"], capture_output=True, text=True)
    print("✓ uv is installed:", ver.stdout.strip())
else:
    print("✗ uv not found — install it with:")
    print("    curl -LsSf https://astral.sh/uv/install.sh | sh   (macOS/Linux)")
    print("    pip install uv                                     (any platform)")

✓ uv is installed: uv 0.11.25 (1fc7de7c4 2026-06-26 aarch64-apple-darwin)


### Step 2 — Create the Virtual Environment

`uv venv` creates an isolated Python environment in `.venv/` (the standard name).  
Always pin `--python` to ensure the same interpreter across all developer machines.

```bash
# Best practice: pin Python version explicitly
uv venv --python 3.11 .venv
```

> `.venv/` belongs in `.gitignore` — never commit it to version control.

In [2]:
import pathlib

PROJECT_ROOT = pathlib.Path(
    "/Users/Guptha/Desktop/interview_kickstart/multimodal_agent"
    "/Live Class Slides + Project/axiomcart-ai-assistant"
)
venv_path = PROJECT_ROOT / ".venv"

if venv_path.exists():
    py_bin = venv_path / "bin" / "python"
    ver = subprocess.run([str(py_bin), "--version"], capture_output=True, text=True)
    print(f"✓ .venv already exists at : {venv_path}")
    print(f"  Python version          : {ver.stdout.strip()}")
else:
    # Create a fresh .venv pinned to Python 3.11
    result = subprocess.run(
        ["uv", "venv", "--python", "3.11", str(venv_path)],
        capture_output=True, text=True, cwd=str(PROJECT_ROOT)
    )
    print(result.stdout or result.stderr)
    print("✓ .venv created")

✓ .venv already exists at : /Users/Guptha/Desktop/interview_kickstart/multimodal_agent/Live Class Slides + Project/axiomcart-ai-assistant/.venv
  Python version          : Python 3.11.13


### Step 3 — Activate the Virtual Environment

Activation puts the venv's `python` and `pip` first on `$PATH`.

```bash
# macOS / Linux
source .venv/bin/activate

# Windows cmd.exe
.venv\Scripts\activate.bat

# Windows PowerShell
.venv\Scripts\Activate.ps1
```

> **Jupyter tip:** when the kernel points to `.venv/bin/python`, activation is implicit.  
> Verify below with `sys.executable`.

In [4]:
import sys, pathlib

exe = pathlib.Path(sys.executable)
print(f"Python executable : {exe}")
print(f"Python version    : {sys.version.split()[0]}")

# Accept any venv-like directory (VS Code creates .venv-1, .venv-2, etc.)
in_venv = any(part.startswith(".venv") for part in exe.parts)
if in_venv:
    print("✓ Kernel is running inside a virtual environment — you're good to go")
else:
    print()
    print("⚠  Kernel is NOT inside a venv. To use the project's .venv:")
    print("   Command Palette → 'Python: Select Interpreter'")
    print(f"   → select: {PROJECT_ROOT / '.venv' / 'bin' / 'python'}")

Python executable : /Users/Guptha/Desktop/interview_kickstart/multimodal_agent/Live Class Slides + Project/axiomcart-ai-assistant/.venv-1/bin/python
Python version    : 3.12.10
✓ Kernel is running inside a virtual environment — you're good to go


### Step 4 — Project Dependencies (`pyproject.toml`)

This project uses `pyproject.toml` (PEP 517/518 standard) — the modern replacement for a bare `requirements.txt`.

| File | Purpose |
|---|---|
| `pyproject.toml` | Declares *what* the project needs (human-authored) |
| `requirements.txt` | Deployment snapshot — pinned flat list |
| `uv.lock` | Hermetic lockfile — every transitive dep, hashed |

The project's `pyproject.toml` is already in the repo. Print its dependency block:

In [5]:
import tomllib

pyproject_path = PROJECT_ROOT / "pyproject.toml"
with open(pyproject_path, "rb") as f:
    data = tomllib.load(f)

proj = data["project"]
print(f"Project  : {proj['name']}  v{proj['version']}")
print(f"Python   : {proj['requires-python']}")
print(f"\nDependencies ({len(proj['dependencies'])}):")
for dep in proj["dependencies"]:
    print(f"  {dep}")

Project  : axiomcart-voice-agent  v0.1.0
Python   : >=3.11

Dependencies (12):
  openai>=1.40
  langchain>=0.3
  langchain-openai>=0.3
  langgraph>=0.2
  chromadb>=0.5
  langchain-chroma>=0.2
  sounddevice>=0.5
  soundfile>=0.12
  numpy>=1.26
  resend>=2.0
  python-dotenv>=1.0
  pydantic>=2.0


### Step 5 — Install All Dependencies

```bash
# Editable install — src/ imports resolve immediately, no reinstall needed after code changes
uv pip install -e .

# Plain requirements.txt alternative
uv pip install -r requirements.txt
```

`-e .` (editable mode) is the standard practice for in-development packages.  
`uv` caches downloaded wheels globally, so subsequent installs are near-instant.

In [25]:
import re

def _clean(text: str) -> str:
    """Strip ANSI escape codes so terminal colour sequences don't bleed into output."""
    return re.sub(r'\x1b\[[0-9;]*[mKGHJF]', '', text)

result = subprocess.run(
    ["uv", "pip", "install", "-e", "."],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT)
)
# Show last 8 lines — the important summary
lines = _clean(result.stdout + result.stderr).strip().splitlines()
for line in lines[-8:]:
    print(line)

status = "✓ All dependencies installed" if result.returncode == 0 else "✗ Install failed"
print(f"\n{status}")

Using Python 3.12.10 environment at: .venv-1
Resolved 108 packages in 570ms
   Building axiomcart-voice-agent @ file:///Users/Guptha/Desktop/interview_kickstart/multimodal_agent/Live%20Class%20Slides%20+%20Project/axiomcart-ai-assistant
      Built axiomcart-voice-agent @ file:///Users/Guptha/Desktop/interview_kickstart/multimodal_agent/Live%20Class%20Slides%20+%20Project/axiomcart-ai-assistant
Prepared 1 package in 519ms
Uninstalled 1 package in 0.89ms
Installed 1 package in 1ms
 ~ axiomcart-voice-agent==0.1.0 (from file:///Users/Guptha/Desktop/interview_kickstart/multimodal_agent/Live%20Class%20Slides%20+%20Project/axiomcart-ai-assistant)

✓ All dependencies installed


### Step 6 — Lock & Export Dependencies

```bash
# Generate hermetic lockfile (commit this to git)
uv lock

# Export pinned requirements for Docker / CI pipelines
uv pip freeze > requirements-lock.txt
```

`uv.lock` captures every transitive dependency with its hash — guarantees identical installs on every machine.  
**Commit** `uv.lock`. **Never commit** `.venv/`.

In [7]:
result = subprocess.run(
    ["uv", "pip", "list", "--format", "columns"],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT)
)
# Filter to only this project's direct deps — avoids noisy output
keywords = {"langchain", "langgraph", "openai", "chroma", "pydantic",
            "dotenv", "sounddevice", "soundfile", "numpy", "axiomcart"}
lines = result.stdout.strip().splitlines()
print(f"{'Package':<40} Version")
print("-" * 55)
for line in lines[2:]:
    name = line.split()[0].lower().replace("-", "")
    if any(k.replace("-", "") in name for k in keywords):
        print(line)

Package                                  Version
-------------------------------------------------------
axiomcart-voice-agent                    0.1.0       /Users/Guptha/Desktop/interview_kickstart/multimodal_agent/Live Class Slides + Project/axiomcart-ai-assistant
chromadb                                 1.5.9
langchain                                1.3.14
langchain-chroma                         1.1.0
langchain-core                           1.4.9
langchain-openai                         1.3.5
langchain-protocol                       0.0.18
langgraph                                1.2.9
langgraph-checkpoint                     4.1.1
langgraph-prebuilt                       1.1.0
langgraph-sdk                            0.4.2
numpy                                    2.5.1
openai                                   2.46.0
pydantic                                 2.13.4
pydantic-core                            2.46.4
pydantic-settings                        2.14.2
python-dotenv        

### Step 7 — Deactivate (when done with this project)

```bash
deactivate
```

Always deactivate before switching to a different project — prevents `$PATH` pollution.

---
## Secrets Management — Loading `.env`

API keys live in `.env` and are loaded via `python-dotenv`.  
**Rules:**
- Never hard-code secrets in source files or notebooks  
- `.env` is listed in `.gitignore` — never commit it  
- Only print masked previews (first 8 + last 4 chars) — never the full value

In [8]:
import os, sys
from dotenv import load_dotenv

# Point explicitly at the project's .env — works regardless of where Jupyter was launched
env_path = PROJECT_ROOT / ".env"
loaded = load_dotenv(dotenv_path=env_path, override=True)

if not loaded:
    raise FileNotFoundError(f".env not found at {env_path}")

# Add project root to sys.path so `src` package is importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Validate required keys — never print values, only masked previews
required = ["OPENAI_API_KEY"]
missing = [k for k in required if not os.environ.get(k)]
if missing:
    raise EnvironmentError(f"Missing required env vars: {missing}")

for key in required:
    val = os.environ[key]
    # Mask: show first 8 chars + "..." + last 4 chars
    masked = val[:8] + "..." + val[-4:] if len(val) > 12 else "***"
    print(f"✓ {key} loaded  (masked: {masked})")

✓ OPENAI_API_KEY loaded  (masked: sk-proj-...L14A)


---
## Stage 1 — Foundation: Config + Data

> **Goal:** Wire up the OpenAI client, embeddings model, and static data store so every later stage has a working foundation to import from.  
> **Files:** `src/config.py` · `src/data.py` · `.env`

**New concept:** environment setup, API clients, logging, dotenv

```
STAGE 1  DATA FLOW
==================

  .env (OPENAI_API_KEY)
       |
       v  load_dotenv()
  config.py
       +-- llm           = ChatOpenAI(model="gpt-4o", temperature=0.3)
       +-- openai_client = OpenAI(api_key=...)   <-- raw SDK, used by voice
       +-- embeddings    = OpenAIEmbeddings("text-embedding-3-small")

  data.py  (static; would be a real DB in production)
       +-- PRODUCT_CATALOG  list[dict]   10 products  --> RAG (Stage 2)
       +-- ORDER_DATABASE   dict         4 orders     --> tools (Stage 4)
       +-- SUPPORT_POLICIES str          policy text  --> prompt (Stage 4)
```

`config.py` calls `sys.exit(1)` at import time if the API key is missing — **fail-fast** principle.  
Real-world: same pattern in Django settings, FastAPI lifespan, and Lambda cold-start init.

In [9]:
# Stage 1 — verify LLM connection
# If this fails: check .env has a valid OPENAI_API_KEY
from src.config import llm

response = llm.invoke("Say hello in exactly five words.")
print("LLM response :", response.content)
print("Model        :", response.response_metadata.get("model_name", "gpt-4o"))
print("\n✓ Stage 1 PASSED — LLM is reachable")

08:50:45 | config             | INFO    | OpenAI clients initialised  (model: gpt-4o, embeddings: text-embedding-3-small)
LLM response : Hello, how are you today?
Model        : gpt-4o-2024-08-06

✓ Stage 1 PASSED — LLM is reachable


In [10]:
# Stage 1 — inspect the data layer (no API calls)
from src.data import PRODUCT_CATALOG, ORDER_DATABASE, SUPPORT_POLICIES

print(f"Products in catalog : {len(PRODUCT_CATALOG)}")
print(f"Orders in database  : {len(ORDER_DATABASE)}")
print(f"Support policy text : {len(SUPPORT_POLICIES)} chars")

print("\nSample product:")
p = PRODUCT_CATALOG[1]   # Bose QC45
print(f"  {p['name']}  ₹{p['price']:,}  ★{p['rating']}  in_stock={p['in_stock']}")

print("\nSample order:")
oid, order = list(ORDER_DATABASE.items())[1]   # ORD102
print(f"  {oid}  {order['customer_name']}  status={order['status']}")

Products in catalog : 8
Orders in database  : 4
Support policy text : 370 chars

Sample product:
  Bose QuietComfort 45 Headphones  ₹10,199  ★4.7  in_stock=True

Sample order:
  ORD102  Priya Patel  status=Delayed


---
## Stage 2 — RAG: Product Search Tool

> **Goal:** Embed the product catalog into a ChromaDB vector store and expose it as a LangChain `@tool` so the LLM can retrieve relevant products via semantic search.  
> **Files:** `src/rag.py` · `src/tools.py` (search_product_catalog only)

**New concept:** vector embeddings, ChromaDB similarity search, `@tool` decorator

```
STAGE 2  RAG PIPELINE
=====================

  PRODUCT_CATALOG (10 products)
         |
         v  _build_documents()  --> LangChain Document per product
  list[Document]  (page_content = structured product text)
         |
         v  OpenAIEmbeddings("text-embedding-3-small")
  1536-dim float vectors  (one per product)
         |
         v  Chroma.from_documents()  --> in-memory collection
  ChromaDB  "axiomcart_products"
         |
         v  .similarity_search(query, k=3)  cosine similarity
  top-3 matching Documents
         |
         v  @tool search_product_catalog(query: str) -> str
  formatted product list  -->  passed to product agent as context
```

> Built once at `import src.tools`, held as a module-level singleton.  
> In production this would be a persistent Pinecone / Weaviate / pgvector collection.

Real-world: same pattern powers Notion AI, GitHub Copilot context retrieval, and enterprise knowledge bases.

In [11]:
# Stage 2 — test RAG search directly (no graph, no LLM reasoning)
# First import triggers embedding of all 10 products (~3s on cold start)
from src.tools import search_product_catalog

queries = [
    "wireless headphones with noise cancellation",
    "budget table fan for home",
    "flagship smartphone with best camera",
]

for q in queries:
    result = search_product_catalog.invoke({"query": q})
    # Show only the first product hit to keep output tidy
    first_hit = result.split("Product 2:")[0].strip()
    print(f"Query : '{q}'")
    print(first_hit[:300])
    print()

print("✓ Stage 2 PASSED — vector search returns semantically relevant products")

08:51:07 | rag                | INFO    | Vector store ready  (8 products indexed)
08:51:07 | tools              | INFO    | search_product_catalog  query='wireless headphones with noise cancellation'
Query : 'wireless headphones with noise cancellation'
Found the following products:

Product 1:
Product: Bose QuietComfort 45 Headphones
Brand: Bose
Category: Electronics
Price: ₹10199
Rating: 4.7/5
Features: Active Noise Cancellation, 24-hour battery, Bluetooth 5.1, USB-C charging
Description: Premium wireless noise-cancelling headphones with legendar

08:51:08 | tools              | INFO    | search_product_catalog  query='budget table fan for home'
Query : 'budget table fan for home'
Found the following products:

Product 1:
Product: Usha Maxx Air 400mm Table Fan
Brand: Usha
Category: Home Appliances
Price: ₹2700
Rating: 4.3/5
Features: 400mm sweep, 3-speed control, Oscillation, Low power consumption
Description: Powerful table fan with superior air delivery and energy efficienc

08:51

---
## Stage 3 — Product Agent Subgraph

> **Goal:** Build a self-contained LangGraph subgraph where the LLM autonomously decides when to call the search tool and loops until it has a complete answer.  
> **Files:** `src/nodes.py` (product_model, product_tools, product_subgraph) · `src/state.py` (AgentState)

**New concept:** LangGraph `StateGraph`, conditional routing, model ⇄ tools loop

```
STAGE 3  PRODUCT AGENT  (internal subgraph)
============================================

  AgentState = { messages: Annotated[list[AnyMessage], operator.add] }

  START
    |
    v
  [model]  <-- product_llm = ChatOpenAI.bind_tools([search_product_catalog])
    |
    +-- tool_calls present? --YES--> [tools]  calls search_product_catalog
    |                                    |
    |                                    +-----------------------------> [model]  (loop)
    |
    +-- no tool_calls --------------------------------------------------------> END

  should_continue() is the conditional edge function.
    "tools" -> route to tools node
    END     -> done, return final message

  LLM decides autonomously:
    greeting   --> no tool call  --> direct reply
    product Q  --> tool call     --> search results --> reasoned reply
```

Real-world: the ReAct (Reason + Act) pattern used by LangChain agents, AutoGPT, and OpenAI Assistants function calling.

In [12]:
# Stage 3 Test A — product question: should call search_product_catalog
from langchain_core.messages import HumanMessage, SystemMessage
from src.nodes import product_subgraph, PRODUCT_PROMPT

result_prod = product_subgraph.invoke({"messages": [
    SystemMessage(content=PRODUCT_PROMPT),
    HumanMessage(content="Show me headphones under ₹15,000"),
]})

# Count how many ToolMessages are in the trace (confirms tool was called)
tool_msgs = [m for m in result_prod["messages"] if hasattr(m, "tool_call_id")]
print(f"Tool calls executed : {len(tool_msgs)}")
print("\nFinal answer:")
print(result_prod["messages"][-1].content[:400])
print()

# Stage 3 Test B — greeting: should NOT call any tool
result_greet = product_subgraph.invoke({"messages": [
    SystemMessage(content=PRODUCT_PROMPT),
    HumanMessage(content="Hey! How are you?"),
]})
tool_msgs_greet = [m for m in result_greet["messages"] if hasattr(m, "tool_call_id")]
print(f"Tool calls for greeting : {len(tool_msgs_greet)}  (expected 0)")
print("Greeting reply:", result_greet["messages"][-1].content[:120])
print("\n✓ Stage 3 PASSED — product subgraph routes tool vs. no-tool correctly")

08:51:18 | nodes              | INFO    | [product:model] tool_calls=True
08:51:18 | nodes              | INFO    | [product:tools] search_product_catalog({'query': 'headphones under ₹15,000'})
08:51:18 | tools              | INFO    | search_product_catalog  query='headphones under ₹15,000'
08:51:22 | nodes              | INFO    | [product:model] tool_calls=False
Tool calls executed : 1

Final answer:
Here are some headphones under ₹15,000 that you might be interested in:

1. **Sony WH-1000XM5 Headphones**
   - **Price:** ₹12,999
   - **Features:** Industry-leading ANC, 30-hour battery life, Speak-to-Chat, LDAC Hi-Res Audio
   - **Description:** Sony's flagship noise-cancelling headphones with exceptional sound quality.
   - **Colors:** Black, Silver
   - **Rating:** 4.9/5
   - **Availability:*

08:51:23 | nodes              | INFO    | [product:model] tool_calls=False
Tool calls for greeting : 0  (expected 0)
Greeting reply: Hello! I'm here and ready to help you find what you need. 

---
## Stage 4 — Support Agent + Tools

> **Goal:** Reuse the exact same subgraph pattern from Stage 3 with a different prompt and tools, proving that agent specialisation is just configuration — not new architecture.  
> **Files:** `src/tools.py` (get_order_status, escalate_to_human) · `src/nodes.py` (support_model, support_tools, support_subgraph)

**New concept:** reusable agent pattern, tool specialisation, HITL groundwork

```
STAGE 4  SUPPORT AGENT  (same topology, different tools)
=========================================================

  product_agent  <-- PRODUCT_PROMPT  +  [search_product_catalog]
  support_agent  <-- SUPPORT_PROMPT  +  [get_order_status, escalate_to_human]

  START --> [model] --+-- tool_calls? --YES--> [tools] --> [model]  (loop)
                      |
                      +-- no tool_calls, no prior tools  --> interrupt()  (Stage 7)
                      |
                      +-- no tool_calls, tools were run  --> END

  get_order_status(identifier)
    - accepts order ID ("ORD102") OR email address
    - normalises "ORD-102" --> "ORD102" internally
    - returns status, ETA, product info from ORDER_DATABASE

  escalate_to_human(reason, customer_name, order_id)
    - appends ticket to ESCALATION_QUEUE
    - optionally sends email via Resend API (RESEND_API_KEY in .env)
```

Design principle: swap prompt + tools → new specialist agent. Same pattern powers Salesforce Einstein, Zendesk AI, and AWS Bedrock agents.

In [13]:
# Stage 4 Test A — call get_order_status tool directly
from src.tools import get_order_status

# Tool normalises "ORD-102" → "ORD102" internally
for identifier in ["ORD101", "ORD102", "ORD-103"]:
    status = get_order_status.invoke({"identifier": identifier})
    # Print first 2 lines only
    lines = status.strip().splitlines()
    print(f"{identifier:8s} → {lines[0]}  |  {lines[2].strip() if len(lines) > 2 else ''}")
print()

08:51:28 | tools              | INFO    | get_order_status  identifier='ORD101'
ORD101   → Order ORD101:  |  Product  : Air Jordan 1 Retro High OG
08:51:28 | tools              | INFO    | get_order_status  identifier='ORD102'
ORD102   → Order ORD102:  |  Product  : Bose QuietComfort 45 Headphones
08:51:28 | tools              | INFO    | get_order_status  identifier='ORD-103'
ORD-103  → Order ORD103:  |  Product  : Usha Maxx Air 400mm Table Fan



In [14]:
# Stage 4 Test B — full support subgraph with order ID in the query
from src.nodes import support_subgraph, SUPPORT_PROMPT

# Providing the order ID upfront prevents the HITL interrupt (tested in Stage 7)
result_support = support_subgraph.invoke({"messages": [
    SystemMessage(content=SUPPORT_PROMPT),
    HumanMessage(content="What is the status of order ORD102?"),
]})

tool_calls = [m for m in result_support["messages"] if hasattr(m, "tool_call_id")]
print(f"Tool calls executed : {len(tool_calls)}  (get_order_status)")
print("\nSupport response:")
print(result_support["messages"][-1].content[:400])
print("\n✓ Stage 4 PASSED — support subgraph calls get_order_status and returns order details")

08:51:33 | nodes              | INFO    | [support:model] tool_calls=True
08:51:33 | nodes              | INFO    | [support:tools] get_order_status({'identifier': 'ORD102'})
08:51:33 | tools              | INFO    | get_order_status  identifier='ORD102'
08:51:34 | nodes              | INFO    | [support:model] tool_calls=False
Tool calls executed : 1  (get_order_status)

Support response:
The status of order ORD102 for Priya Patel is currently delayed. The order, which includes the Bose QuietComfort 45 Headphones priced at ₹10,199, was placed on February 8, 2026. The estimated time of arrival was initially set for February 15, 2026, but it has been delayed due to bad weather conditions in the transit region.

✓ Stage 4 PASSED — support subgraph calls get_order_status and returns order details


---
## Stage 5 — Orchestrator + Multi-Agent Routing

> **Goal:** Use GPT-4o's structured output to classify every query and dispatch it to the right agent(s) via `Send()` — the biggest conceptual leap in the whole system.  
> **Files:** `src/state.py` (ClassificationResult, AgentTask, WorkerInput) · `src/nodes.py` (orchestrator_node)

**New concept:** Pydantic structured LLM output, `Send()` parallel fan-out

```
STAGE 5  ORCHESTRATOR CLASSIFICATION + DISPATCH
================================================

  user_query (str)
       |
       v  llm.with_structured_output(ClassificationResult)
       |  GPT-4o is forced to return a valid Pydantic model

  ClassificationResult
  +-- tasks: list[AgentTask]
  |     AgentTask(agent="product_agent", task_description="...")
  |     AgentTask(agent="support_agent",  task_description="...")
  +-- requires_synthesis: bool
  +-- reasoning: str

       |
       v  routing rules
       +-- greeting / chitchat  --> Send("product_agent", payload)
       +-- product-only         --> Send("product_agent", payload)
       +-- support-only         --> Send("support_agent",  payload)
       +-- mixed                --> Send("product_agent") + Send("support_agent")
                                    both execute in parallel (asyncio event loop)

  Send(node, state) = LangGraph's fan-out primitive
    - targets a specific node by name
    - passes isolated state to each worker
    - results are merged back via agent_results_reducer (operator.add)
```

Real-world: same fan-out pattern in AWS Step Functions, Temporal workflows, and Google Cloud Workflows.

In [15]:
# Stage 5 — test orchestrator classification for all 3 query archetypes
from src.config import llm
from src.state import ClassificationResult

classifier = llm.with_structured_output(ClassificationResult)

test_cases = [
    ("greeting",      "Hi! What can AxiomCart help me with?"),
    ("product-only",  "Show me wireless headphones under ₹15,000"),
    ("support-only",  "My order ORD102 hasn't arrived — what's happening?"),
    ("mixed",         "ORD102 is late, show me headphone alternatives in the same price range"),
]

print(f"{'Type':<15}  {'Agents Assigned':<40}  Synthesis")
print("-" * 65)
for label, query in test_cases:
    result = classifier.invoke(
        f"Classify this customer query.\nQUERY: {query}"
    )
    agents = [t.agent.replace("_agent", "") for t in result.tasks]
    print(f"{label:<15}  {str(agents):<40}  {result.requires_synthesis}")

print("\n✓ Stage 5 PASSED — orchestrator classifies all query types correctly")

Type             Agents Assigned                           Synthesis
-----------------------------------------------------------------
greeting         ['product']                               False
product-only     ['product']                               False
support-only     ['support']                               False
mixed            ['support', 'product']                    True

✓ Stage 5 PASSED — orchestrator classifies all query types correctly


---
## Stage 6 — Synthesizer + Full Graph

> **Goal:** Wire all nodes into a compiled `StateGraph`, add a synthesizer that merges parallel agent results into one reply, and attach `MemorySaver` for cross-turn state persistence.  
> **Files:** `src/graph.py` · `src/nodes.py` (product_agent, support_agent, synthesizer_node)

**New concept:** response synthesis, `StateGraph` compilation, `MemorySaver` checkpointer

```
STAGE 6  FULL COMPILED GRAPH
=============================

  START
    |
    v
  [orchestrator]  classifies + Command(goto=[Send(...), Send(...)])
    |
    +----------------------------------+
    v                                  v
  [product_agent]             [support_agent]
  (product_subgraph)          (support_subgraph)
    |  Command(update=agent_results,   |
    |          goto="synthesizer")     |
    +------------- agent_results_reducer (operator.add) -----------+
                                       |
                                       v
                               [synthesizer]
                                       |
                          +-----------+-----------+
                          |                       |
                    1 result?               multiple?
                    pass-through        LLM merges into
                    (no extra cost)     one coherent reply
                          |
                          v
                     final_answer (str)
                          |
                          v
                         END

  MemorySaver:  snapshots AxiomCartState after every node
  thread_id:    one per user session  --> isolated history
  Production:   replace with SqliteSaver or RedisSaver
```

In [16]:
# Stage 6 Test A — product-only (single agent → pass-through in synthesizer)
import uuid
from langchain_core.messages import HumanMessage
from src.graph import axiomcart_graph

cfg_a = {"configurable": {"thread_id": f"wt-s6-a-{uuid.uuid4().hex[:6]}"}}
result_a = axiomcart_graph.invoke(
    {"messages": [HumanMessage(content="Show me noise-cancelling headphones")],
     "user_query": "Show me noise-cancelling headphones"},
    cfg_a
)
print("=== Product-only query ===")
print(f"Agents used : {[t.agent for t in result_a.get('tasks', [])]}")
print(f"Synthesis   : {result_a.get('requires_synthesis')}")
print("\nAnswer:")
print(result_a["final_answer"][:350])

08:51:49 | graph              | INFO    | Graph compiled  (with MemorySaver for conversation persistence)
08:51:49 | nodes              | INFO    | Orchestrator  query='Show me noise-cancelling headphones'
08:51:50 | nodes              | INFO    |   routing=['product_agent']  synthesis=False
08:51:50 | nodes              | INFO    | Product Agent  task='Assist the customer in finding noise-cancelling headphones by providing product recommendations, details, and availability.'
08:51:51 | nodes              | INFO    | [product:model] tool_calls=True
08:51:51 | nodes              | INFO    | [product:tools] search_product_catalog({'query': 'noise-cancelling headphones'})
08:51:51 | tools              | INFO    | search_product_catalog  query='noise-cancelling headphones'
08:51:54 | nodes              | INFO    | [product:model] tool_calls=False
08:51:54 | nodes              | INFO    | Synthesizer  single-agent pass-through
=== Product-only query ===
Agents used : ['product_agent']
Synth

In [17]:
# Stage 6 Test B — mixed query: both agents run in parallel, synthesizer merges
cfg_b = {"configurable": {"thread_id": f"wt-s6-b-{uuid.uuid4().hex[:6]}"}}
result_b = axiomcart_graph.invoke(
    {"messages": [HumanMessage(content="ORD102 is late, show me headphone alternatives")],
     "user_query": "ORD102 is late, show me headphone alternatives"},
    cfg_b
)
print("=== Mixed query (product + support → synthesizer) ===")
print(f"Agents used         : {[t.agent for t in result_b.get('tasks', [])]}")
print(f"Synthesis required  : {result_b.get('requires_synthesis')}")
print(f"Agent result count  : {len(result_b.get('agent_results', []))}")
print("\nMerged answer:")
print(result_b["final_answer"][:500])
print("\n✓ Stage 6 PASSED — full graph wired, synthesizer merges parallel results")

08:51:59 | nodes              | INFO    | Orchestrator  query='ORD102 is late, show me headphone alternatives'
08:52:01 | nodes              | INFO    |   routing=['support_agent', 'product_agent']  synthesis=True
08:52:01 | nodes              | INFO    | Support Agent  task='Check the status of order ORD102 and provide an update to the customer.'
08:52:01 | nodes              | INFO    | Product Agent  task='Provide alternative headphone recommendations to the customer.'
08:52:02 | nodes              | INFO    | [support:model] tool_calls=True
08:52:02 | nodes              | INFO    | [support:tools] get_order_status({'identifier': 'ORD102'})
08:52:02 | tools              | INFO    | get_order_status  identifier='ORD102'
08:52:02 | nodes              | INFO    | [product:model] tool_calls=True
08:52:02 | nodes              | INFO    | [product:tools] search_product_catalog({'query': 'headphones'})
08:52:02 | tools              | INFO    | search_product_catalog  query='headphones'
08:

In [18]:
# Stage 6 Bonus — visualise the compiled graph topology as Mermaid
# Paste the output into https://mermaid.live to see the rendered diagram
print(axiomcart_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	orchestrator(orchestrator)
	product_agent(product_agent)
	support_agent(support_agent)
	synthesizer(synthesizer)
	__end__([<p>__end__</p>]):::last
	__start__ --> orchestrator;
	orchestrator -.-> product_agent;
	orchestrator -.-> support_agent;
	orchestrator -.-> synthesizer;
	product_agent -.-> synthesizer;
	support_agent -.-> synthesizer;
	synthesizer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



---
## Stage 7 — Human-in-the-Loop (HITL)

> **Goal:** Let the graph pause mid-execution to collect a missing detail from the user, then resume from the exact checkpoint — without losing any conversation state.  
> **Files:** `src/nodes.py` (support_model — interrupt logic) · `src/graph.py` (MemorySaver)

**New concept:** `interrupt()`, `Command(resume=...)`, graph pause and resume

```
STAGE 7  HITL  PAUSE / RESUME FLOW
====================================

  User: "Where is my order?"  (no order ID given)
         |
         v
  [orchestrator] --> Send("support_agent")
         |
         v
  [support_model]
         |  LLM: "Please provide your order ID or email"
         |  No tool calls yet -> graph cannot proceed
         |
         v  interrupt(question)      LangGraph PAUSES here
                                     Full AxiomCartState saved to MemorySaver

  Caller receives:
    { "__interrupt__": [Interrupt(value="Please provide...")] }

         |  User types "ORD102"
         |
         v  axiomcart_graph.invoke(Command(resume="ORD102"), config)
                                   graph RESUMES from checkpoint

  [support_model]
         |  HumanMessage("ORD102") appended to messages
         |  LLM now has the ID -> calls get_order_status("ORD102")
         v
  [support_tools] --> [support_model] --> [synthesizer] --> final_answer

  Same thread_id = same checkpoint = same conversation history across pauses.
  Real-world: approval gates, multi-step forms, escalation confirmation.
```

In [19]:
# Stage 7 — HITL: vague support query triggers interrupt, then resume with order ID
from langgraph.types import Command

# Use a fresh thread_id so we start with a clean checkpoint
cfg_hitl = {"configurable": {"thread_id": f"wt-s7-{uuid.uuid4().hex[:6]}"}}

# Turn 1: vague query — support agent cannot look up anything without an order ID
result = axiomcart_graph.invoke(
    {"messages": [HumanMessage(content="Where is my order?")],
     "user_query": "Where is my order?"},
    cfg_hitl
)

if "__interrupt__" in result and result["__interrupt__"]:
    question = result["__interrupt__"][0].value
    print("=== Graph PAUSED ===")
    print(f"Agent asks : {question[:150]}")
    print()

    # Turn 2: user provides the missing order ID → graph resumes from checkpoint
    result = axiomcart_graph.invoke(Command(resume="ORD102"), cfg_hitl)

    print("=== Graph RESUMED ===")
    print(result["final_answer"][:400])
else:
    # Agent may answer directly if it inferred enough context
    print("Agent responded without HITL interrupt:")
    print(result.get("final_answer", "")[:300])

print("\n✓ Stage 7 PASSED — HITL interrupt/resume works correctly")

08:52:15 | nodes              | INFO    | Orchestrator  query='Where is my order?'
08:52:17 | nodes              | INFO    |   routing=['support_agent']  synthesis=False
08:52:17 | nodes              | INFO    | Support Agent  task="Check the status of the customer's order and provide an update."
08:52:17 | nodes              | INFO    | [support:model] tool_calls=False
08:52:17 | nodes              | INFO    | [support:model] HITL: interrupting to collect user info


Deserializing unregistered type src.state.AgentTask from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('src.state', 'AgentTask')]


=== Graph PAUSED ===
Agent asks : Could you please provide your order ID (e.g. ORD101) or registered email address so I can look up your order?

08:52:17 | nodes              | INFO    | Support Agent  task="Check the status of the customer's order and provide an update."
08:52:18 | nodes              | INFO    | [support:model] tool_calls=False
08:52:18 | nodes              | INFO    | [support:model] HITL: interrupting to collect user info
08:52:18 | nodes              | INFO    | [support:model] HITL: user replied 'ORD102'
08:52:19 | nodes              | INFO    | [support:model] tool_calls=True
08:52:19 | nodes              | INFO    | [support:tools] get_order_status({'identifier': 'ORD102'})
08:52:19 | tools              | INFO    | get_order_status  identifier='ORD102'
08:52:20 | nodes              | INFO    | [support:model] tool_calls=False
08:52:20 | nodes              | INFO    | Synthesizer  single-agent pass-through
=== Graph RESUMED ===
Your order for the Bose QuietComfor

---
## Stage 8 — Interactive REPL

> **Goal:** Wrap the graph in an `AxiomCartAssistant` class with session management so a user can hold a natural multi-turn conversation across many queries using the same checkpointed thread.  
> **Files:** `src/main.py`

**New concept:** session management, `thread_id`, multi-turn conversation

```
STAGE 8  AxiomCartAssistant  SESSION WRAPPER
=============================================

  AxiomCartAssistant
  |
  +-- thread_id = uuid4().hex   unique per session
  |                              MemorySaver keys state by this ID
  |
  +-- query(text: str) -> str
  |     1. invoke graph with HumanMessage(text) + thread config
  |     2. if "__interrupt__": collect answer -> Command(resume=...) loop
  |     3. return final_answer
  |
  +-- text_loop()    REPL: input() -> query() -> print  (until "quit")
  |
  +-- voice_loop()   mic -> Whisper -> query() -> TTS speaker  (Stage 9)

  Multi-turn: MemorySaver stores the full message list per thread_id.
  Turn 2 sees [Turn1_human, Turn1_ai, Turn2_human] — full history, no tricks.

Terminal commands:
  uv run python -m src.main              # interactive text REPL
  uv run python -m src.main --query "…"  # single query, exits
  uv run python -m src.main --voice      # voice loop (Stage 9)
```

In [20]:
# Stage 8 — single-query mode via AxiomCartAssistant
from src.main import AxiomCartAssistant

# enable_voice=False → no mic/speaker needed, pure text path
assistant = AxiomCartAssistant(enable_voice=False)

answer = assistant.query("What Sony products do you carry?")
print("Query  : What Sony products do you carry?")
print("Answer :", answer[:400])
print(f"\nSession thread_id : {assistant.thread_id}")

08:52:27 | main               | INFO    | Assistant ready  (voice=False)
08:52:27 | nodes              | INFO    | Orchestrator  query='What Sony products do you carry?'
08:52:28 | nodes              | INFO    |   routing=['product_agent']  synthesis=False
08:52:28 | nodes              | INFO    | Product Agent  task='Provide information on the Sony products available in the catalog.'
08:52:29 | nodes              | INFO    | [product:model] tool_calls=True
08:52:29 | nodes              | INFO    | [product:tools] search_product_catalog({'query': 'Sony products'})
08:52:29 | tools              | INFO    | search_product_catalog  query='Sony products'
08:52:31 | nodes              | INFO    | [product:model] tool_calls=False
08:52:31 | nodes              | INFO    | Synthesizer  single-agent pass-through
Query  : What Sony products do you carry?
Answer : We currently have the following Sony product available:

- **Sony WH-1000XM5 Headphones**
  - **Price:** ₹12,999
  - **Rating:** 4.9/5

In [21]:
# Stage 8 — multi-turn: Turn 2 references context from Turn 1
# Both turns use the same assistant instance → same thread_id → shared history

answer1 = assistant.query("Show me wireless headphones under ₹15,000")
print("Turn 1:")
print(answer1[:200])
print()

# This follow-up works because the graph sees both turns in its message list
answer2 = assistant.query("Which one has the best battery life?")
print("Turn 2 (follow-up, no context needed):")
print(answer2[:250])
print("\n✓ Stage 8 PASSED — multi-turn conversation with persistent history")

08:52:36 | nodes              | INFO    | Orchestrator  query='Show me wireless headphones under ₹15,000'
08:52:37 | nodes              | INFO    |   routing=['product_agent']  synthesis=False
08:52:37 | nodes              | INFO    | Product Agent  task='Assist the customer in finding wireless headphones priced under ₹15,000. Provide recommendations and details about available products within this price range.'
08:52:39 | nodes              | INFO    | [product:model] tool_calls=True
08:52:39 | nodes              | INFO    | [product:tools] search_product_catalog({'query': 'Sony wireless headphones under ₹15,000'})
08:52:39 | tools              | INFO    | search_product_catalog  query='Sony wireless headphones under ₹15,000'
08:52:41 | nodes              | INFO    | [product:model] tool_calls=False
08:52:41 | nodes              | INFO    | Synthesizer  single-agent pass-through
Turn 1:
Here are some Sony wireless headphones under ₹15,000 that you might be interested in:

1. **Sony WH

---
## Stage 9 — Voice I/O

> **Goal:** Replace text I/O with a microphone → Whisper → graph → TTS → speaker pipeline, making the exact same multi-agent system accessible via spoken conversation.  
> **Files:** `src/voice.py` · `src/main.py` (voice_loop)

**New concept:** OpenAI Whisper STT, TTS API, `sounddevice` audio recording

```
STAGE 9  VOICE PIPELINE
========================

  +----------------------------------------------+
  |  VoiceRecorder                               |
  |                                              |
  |  sounddevice.rec(duration, sr=16000)         |  <-- PCM from microphone
  |  soundfile.write() --> WAV buffer            |
  |  openai.audio.transcriptions.create()        |  <-- Whisper API
  |  --> transcript (str)                        |
  +--------------------+-------------------------+
                       |
                       v  transcript
         AxiomCartAssistant.query(transcript)
                       |
                       v  final_answer (str)
  +----------------------------------------------+
  |  VoiceSpeaker                                |
  |                                              |
  |  openai.audio.speech.create(                 |  <-- TTS API (tts-1)
  |      model="tts-1", voice="nova")            |
  |  --> MP3 file                                |
  |  soundfile.read() + sounddevice.play()       |  <-- speaker output
  +----------------------------------------------+

  Voices available: alloy  echo  fable  onyx  nova  shimmer
  Whisper: auto-detects language, supports English, Hindi, 50+ more
```

---
### Recording live audio in this notebook

`sounddevice` records synchronously in a Jupyter cell — **the cell blocks until done**.  
There are two valid paths:

| Path | When to use |
|---|---|
| **Notebook cell** (cells below) | Quick demo, mic already granted to the kernel process |
| **Terminal** `uv run python -m src.main --voice` | Full interactive loop, clean I/O |

**macOS mic permission (one-time setup):**  
The kernel process inherits permissions from the app that launched Jupyter.  
If recording fails → open **System Settings → Privacy → Microphone** → enable VS Code (or Terminal).

```
Notebook recording flow
=======================

  [record cell]
       |
       v  sounddevice.rec(duration=5, sr=16000)  -- blocking, 5 seconds
       v  soundfile.write() --> WAV bytes in memory
       |
       v  IPython.display.Audio(wav_bytes)        -- plays back in notebook
       |
       v  openai.audio.transcriptions.create()   -- Whisper STT
       |
       v  transcript (str) --> query the agent
       |
       v  final_answer --> display in notebook
```

In [26]:
# Stage 9A — TTS: synthesise speech and play it back via IPython Audio widget
# No microphone needed.  IPython.display.Audio embeds a playable widget in the notebook.
import os
from IPython.display import Audio, display
from src.voice import VoiceSpeaker

speaker = VoiceSpeaker(voice="nova", speed=1.0)

test_text = "Welcome to AxiomCart. I can help you find products and check your orders."
mp3_path = speaker.synthesise(test_text)   # returns path to temp MP3 file

if mp3_path and os.path.exists(mp3_path):
    size_kb = os.path.getsize(mp3_path) / 1024
    print(f"  Text   : {test_text}")
    print(f"  Voice  : nova  (tts-1)")
    print(f"  Size   : {size_kb:.1f} KB")
    print()
    # Embed the audio widget — click Play to hear it
    display(Audio(mp3_path, autoplay=False))
else:
    print("TTS failed — check OPENAI_API_KEY")

print("\n✓ Stage 9A PASSED — TTS works, audio widget embedded above")

09:01:58 | voice              | INFO    | TTS saved → /var/folders/9z/9w8rqc1x66z0cf34rm2sr1wc0000gp/T/tts_83bba7f7.mp3
  Text   : Welcome to AxiomCart. I can help you find products and check your orders.
  Voice  : nova  (tts-1)
  Size   : 88.6 KB




✓ Stage 9A PASSED — TTS works, audio widget embedded above


In [27]:
# Stage 9B — Live microphone recording in the notebook
#
# BEFORE RUNNING:
#   macOS: System Settings > Privacy & Security > Microphone > enable VS Code / Terminal
#   Linux: microphone device must be accessible to the Jupyter process
#   Windows: should work out-of-the-box if mic permissions are granted
#
# This cell records for RECORD_SECONDS, transcribes with Whisper,
# queries the agent, and speaks the answer back via TTS.
# The cell BLOCKS while recording — you will see "Recording..." in the output.

import io
import numpy as np
import sounddevice as sd
import soundfile as sf
from IPython.display import Audio, display

RECORD_SECONDS = 5   # change to 8 or 10 for longer queries
SAMPLE_RATE    = 16_000

# ── Step 1: check mic availability ───────────────────────────────────────
try:
    device_info = sd.query_devices(kind="input")
    print(f"Microphone detected : {device_info['name']}")
    mic_available = True
except Exception as e:
    print(f"No microphone found ({e})")
    print("Skipping live recording — run  uv run python -m src.main --voice  in a terminal instead.")
    mic_available = False

if mic_available:
    # ── Step 2: record ───────────────────────────────────────────────────
    print(f"\nRecording for {RECORD_SECONDS}s — speak now ...")
    audio = sd.rec(
        int(RECORD_SECONDS * SAMPLE_RATE),
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32",
    )
    sd.wait()   # blocks until done
    print("Recording complete.")

    # ── Step 3: play back in notebook so you can verify the capture ──────
    wav_buf = io.BytesIO()
    sf.write(wav_buf, audio, SAMPLE_RATE, format="WAV")
    wav_buf.seek(0)
    print("\nPlayback of what was recorded:")
    display(Audio(wav_buf.read(), rate=SAMPLE_RATE, autoplay=False))

    # ── Step 4: transcribe with Whisper ──────────────────────────────────
    from src.config import openai_client
    wav_buf.seek(0)
    wav_buf.name = "recording.wav"
    result_stt = openai_client.audio.transcriptions.create(
        model="whisper-1", file=wav_buf, language="en"
    )
    transcript = result_stt.text
    print(f"\nWhisper transcript : \"{transcript}\"")

    if transcript.strip():
        # ── Step 5: query the agent ──────────────────────────────────────
        from src.main import AxiomCartAssistant
        voice_session = AxiomCartAssistant(enable_voice=False)
        answer = voice_session.query(transcript)
        print(f"\nAgent answer : {answer[:300]}")

        # ── Step 6: speak the answer back ────────────────────────────────
        tts_path = speaker.synthesise(answer[:500])   # keep TTS short
        if tts_path:
            print("\nAgent audio response:")
            display(Audio(tts_path, autoplay=False))
        print("\n✓ Stage 9B PASSED — full mic -> Whisper -> agent -> TTS loop works")
    else:
        print("Transcript was empty — try again with clearer audio or increase RECORD_SECONDS")

Microphone detected : MacBook Pro Microphone

Recording for 5s — speak now ...
Recording complete.

Playback of what was recorded:



Whisper transcript : "Thank you for watching."
09:02:11 | main               | INFO    | Assistant ready  (voice=False)
09:02:11 | nodes              | INFO    | Orchestrator  query='Thank you for watching.'
09:02:12 | nodes              | INFO    |   routing=['product_agent']  synthesis=False
09:02:12 | nodes              | INFO    | Product Agent  task="Respond to the customer's expression of gratitude and engage in general conversation."
09:02:13 | nodes              | INFO    | [product:model] tool_calls=False
09:02:13 | nodes              | INFO    | Synthesizer  single-agent pass-through

Agent answer : You're welcome! If there's anything else you need or if you have any questions, feel free to ask. How can I assist you today?
09:02:16 | voice              | INFO    | TTS saved → /var/folders/9z/9w8rqc1x66z0cf34rm2sr1wc0000gp/T/tts_d8a591f8.mp3

Agent audio response:



✓ Stage 9B PASSED — full mic -> Whisper -> agent -> TTS loop works


---
## Final Integration Smoke Test

Runs all 9 stages end-to-end in one pass — one query per archetype.

```
COMPLETE SYSTEM — STAGE MAP
============================

  Stage 1  config.py + data.py          ──  Foundation (API clients, data layer)
  Stage 2  rag.py + tools.py            ──  RAG (ChromaDB, @tool decorator)
  Stage 3  nodes.py (product subgraph)  ──  Product Agent (model ⇄ tools loop)
  Stage 4  tools.py + support subgraph  ──  Support Agent (order tools, HITL prep)
  Stage 5  state.py + orchestrator      ──  Routing (Pydantic structured output, Send())
  Stage 6  synthesizer + graph.py       ──  Full Graph (MemorySaver, synthesizer)
  Stage 7  interrupt() + MemorySaver    ──  HITL (pause / resume)
  Stage 8  main.py (REPL + assistant)   ──  Session management (thread_id, multi-turn)
  Stage 9  voice.py (Whisper + TTS)     ──  Voice I/O (spoken interface)
```

In [24]:
# Final integration smoke test — exercises the complete pipeline
from src.main import AxiomCartAssistant

session = AxiomCartAssistant(enable_voice=False)

test_cases = [
    ("chitchat",      "Hi! What can you help me with today?"),
    ("product",       "Recommend a premium smartphone under ₹1.5 lakh"),
    ("support",       "Check the status of order ORD103 for me"),
    ("mixed",         "ORD102 is delayed — also show me alternatives to the product I ordered"),
]

for label, query in test_cases:
    print(f"\n{'='*55}")
    print(f"  [{label.upper():10s}]  {query}")
    print(f"{'='*55}")
    answer = session.query(query)
    print(answer[:350])

print("\n\n" + "="*55)
print("  ALL STAGES VERIFIED")
print("  AxiomCart multi-agent system is fully operational")
print("="*55)

08:53:21 | main               | INFO    | Assistant ready  (voice=False)

  [CHITCHAT  ]  Hi! What can you help me with today?
08:53:21 | nodes              | INFO    | Orchestrator  query='Hi! What can you help me with today?'
08:53:23 | nodes              | INFO    |   routing=['product_agent']  synthesis=False
08:53:23 | nodes              | INFO    | Product Agent  task='Engage with the customer by responding to their greeting and offering assistance with general inquiries or product-related questions.'
08:53:24 | nodes              | INFO    | [product:model] tool_calls=False
08:53:24 | nodes              | INFO    | Synthesizer  single-agent pass-through
Hi there! I'm here to help you find and learn about products available at AxiomCart. Whether you're looking for something specific or just browsing, feel free to ask. How can I assist you today?

  [PRODUCT   ]  Recommend a premium smartphone under ₹1.5 lakh
08:53:24 | nodes              | INFO    | Orchestrator  query='Recommend